# Derf Messenger — Android Build

This notebook builds the Derf Android APK on Google Colab (Linux).

**What it does:**
1. Clones your Derf project
2. Installs buildozer + python-for-android
3. Accepts SDK licenses
4. Builds debug APK
5. Downloads APK to your computer

**Your phone must be connected via USB and detected by ADB.**

## 1. Mount Google Drive (to save APK)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
APK_OUTPUT = '/content/drive/MyDrive/derf.apk'
print('Google Drive mounted. APK will be saved to:', APK_OUTPUT)

## 2. Upload Your Derf Files

Upload these files from `C:\Users\GG\Desktop\Derf\`:
- `Derf.py`
- `derf_bg.py`
- `buildozer.spec`
- `buildozer_spec_additions.txt`

In [ ]:
from google.colab import files
import os

os.makedirs('/content/derf', exist_ok=True)
os.chdir('/content/derf')
print('Upload Derf.py, derf_bg.py, buildozer.spec, buildozer_spec_additions.txt')
uploaded = files.upload()
for fn in uploaded.keys():
    print(f'Uploaded: {fn} ({len(uploaded[fn])} bytes)')

## 3. Also Upload Any Extra Python Dependencies

If `Derf.py` imports any modules not in standard Python, upload them too.

In [ ]:
# Upload any extra .py files if needed
extra = files.upload()
for fn in extra.keys():
    print(f'Uploaded extra: {fn}')

## 4. Install Build System

In [ ]:
!apt update -qq && apt install -qq -y python3-pip python3-setuptools git zip unzip openjdk-17-jdk-headless ant autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake 2>&1 | tail -5
print('System packages installed')

In [ ]:
!pip install --upgrade pip setuptools wheel 2>&1 | tail -3
!pip install buildozer==1.6.0 python-for-android==2024.1.21 cython 2>&1 | tail -5
!pip show buildozer python-for-android 2>&1 | grep -E '^(Name|Version)'

## 5. Verify Files

In [ ]:
import os
os.chdir('/content/derf')
for f in ['Derf.py', 'derf_bg.py', 'buildozer.spec', 'buildozer_spec_additions.txt']:
    if os.path.exists(f):
        print(f'  OK: {f} ({os.path.getsize(f)} bytes)')
    else:
        print(f'  MISSING: {f}')

## 6. Build the APK

This takes **15-30 minutes** on Colab. Go get coffee.

In [ ]:
# Accept all Android SDK licenses
!mkdir -p ~/.android && echo '24333f8a63b6825ea9c5514f83c2829b004d1fee' > ~/.android/repositories.cfg
!yes | buildozer android debug 2>&1 | tee build.log
print('=== BUILD COMPLETE ===')

## 7. Find and Copy APK

In [ ]:
import os
import glob

# Find APK
apks = glob.glob('/content/derf/.buildozer/android/**/bin/*.apk', recursive=True)
if apks:
    print(f'Found APK: {apks[-1]}')
    apk = apks[-1]
    size = os.path.getsize(apk)
    print(f'Size: {size / 1024 / 1024:.1f} MB')

    # Copy to Drive
    import shutil
    shutil.copy(apk, APK_OUTPUT)
    print(f'APK saved to: {APK_OUTPUT}')
    
    # Provide download link
    from google.colab import files
    files.download(apk)
else:
    print('APK not found! Check build.log for errors')
    print('Last 30 lines of build.log:')
    !tail -30 build.log